<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_SourceMappingMetricFix_v11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIRy–pyOMOP Transformation Fidelity

Source mapping coverage correction.

This notebook reuses the completed v10 corrected TFL audits. It does not rerun FHIRy, pyOMOP, Encounter restoration, warning detection, or database generation.

Only `source_mapping_coverage` is corrected to use unique source resources rather than audit-event rows. The source-resource key is `(source_resource_type, source_df_index)` so V2 duplicate Encounter IDs are not collapsed.

# Phase A

## Load v10 corrected audits

In [1]:
from pathlib import Path
import shutil
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
RUN_ROOT = MYDRIVE / "fhir_omop_colab" / "tfl_execution_v6"
V10_ROOT = RUN_ROOT / "encounter_audit_repair_v10"

AUDIT_DIR = V10_ROOT / "fidelity_audit_corrected"
V10_OUTPUT_DIR = V10_ROOT / "outputs"
V11_OUTPUT_DIR = V10_ROOT / "metric_fix_v11"
V11_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTS = ["V0", "V1", "V2", "V3", "V4", "V5"]

CORRECTED_AUDIT = {}

for variant in VARIANTS:
    path = AUDIT_DIR / f"{variant}_fidelity_audit_v10.parquet"

    if not path.exists():
        raise FileNotFoundError(path)

    audit = pd.read_parquet(path)

    required = {
        "source_resource_type",
        "source_df_index",
        "is_mapping_event",
        "target_omop_record_id",
    }

    missing = required - set(audit.columns)

    if missing:
        raise RuntimeError(
            f"{variant}: missing required audit columns: "
            + ", ".join(sorted(missing))
        )

    CORRECTED_AUDIT[variant] = audit


v10_metrics_path = V10_OUTPUT_DIR / "tfl_primary_metrics_v10.csv"

if not v10_metrics_path.exists():
    raise FileNotFoundError(v10_metrics_path)

V10_METRICS = pd.read_csv(v10_metrics_path)

display(V10_METRICS)

Mounted at /content/drive


,variant,audit_items,mapped_items,lineage_coverage,source_mapping_coverage,transformation_loss_rate,ambiguity_warning_rate
0,V0,179333,151071,1.00000,0.842405,0.157595,0.211612
1,V1,179333,151071,1.00000,0.842405,0.157595,0.212510
2,V2,179333,151071,0.98416,0.842405,0.170939,0.224956
3,V3,179333,151071,1.00000,0.842405,0.157595,0.225608
4,V4,176507,151071,1.00000,0.855892,0.144108,0.199873
5,V5,179333,151071,1.00000,0.842405,0.157595,0.226506


# Phase B

## Correct source mapping coverage

In [2]:
SOURCE_KEY = [
    "source_resource_type",
    "source_df_index",
]

EXPECTED_SOURCE_RESOURCES = {
    "V0": 154333,
    "V1": 154333,
    "V2": 154333,
    "V3": 154333,
    "V4": 151507,
    "V5": 154333,
}


def calculate_resource_level_mapping(audit, variant):
    work = audit.copy()

    incomplete_key = (
        work["source_resource_type"].isna()
        | work["source_df_index"].isna()
    )

    if incomplete_key.any():
        raise RuntimeError(
            f"{variant}: {int(incomplete_key.sum()):,} audit rows "
            "have an incomplete source-resource key."
        )

    work["source_resource_type"] = (
        work["source_resource_type"].astype(str)
    )

    work["source_df_index"] = pd.to_numeric(
        work["source_df_index"],
        errors="raise",
    ).astype("int64")

    work["mapped_event"] = (
        work["is_mapping_event"]
        .fillna(False)
        .astype(bool)
        & work["target_omop_record_id"].notna()
    )

    resource_level = (
        work.groupby(
            SOURCE_KEY,
            as_index=False,
            sort=False,
        )
        .agg(
            audit_events=("mapped_event", "size"),
            mapped=("mapped_event", "max"),
        )
    )

    total = len(resource_level)
    mapped = int(resource_level["mapped"].sum())
    unmapped = total - mapped

    return {
        "variant": variant,
        "source_resources_total": total,
        "source_resources_mapped": mapped,
        "source_resources_unmapped": unmapped,
        "multi_event_source_resources": int(
            (resource_level["audit_events"] > 1).sum()
        ),
        "source_mapping_coverage_corrected": (
            mapped / total if total else np.nan
        ),
    }


SOURCE_MAPPING_V11 = pd.DataFrame([
    calculate_resource_level_mapping(
        CORRECTED_AUDIT[variant],
        variant,
    )
    for variant in VARIANTS
])

display(SOURCE_MAPPING_V11)


for variant, expected in EXPECTED_SOURCE_RESOURCES.items():
    actual = int(
        SOURCE_MAPPING_V11.loc[
            SOURCE_MAPPING_V11["variant"] == variant,
            "source_resources_total",
        ].iloc[0]
    )

    if actual != expected:
        raise RuntimeError(
            f"{variant}: expected {expected:,} unique source resources, "
            f"found {actual:,}."
        )


if not (
    SOURCE_MAPPING_V11["source_mapping_coverage_corrected"]
    .between(0.0, 1.0)
    .all()
):
    raise RuntimeError(
        "Corrected source mapping coverage is outside [0, 1]."
    )

print("PASS: resource-level source mapping coverage calculated.")

,variant,source_resources_total,source_resources_mapped,source_resources_unmapped,multi_event_source_resources,source_mapping_coverage_corrected
0,V0,154333,126071,28262,25000,0.816876
1,V1,154333,126071,28262,25000,0.816876
2,V2,154333,126071,28262,25000,0.816876
3,V3,154333,126071,28262,25000,0.816876
4,V4,151507,126071,25436,25000,0.832113
5,V5,154333,126071,28262,25000,0.816876


PASS: resource-level source mapping coverage calculated.


## Compare old and corrected denominator

In [4]:
# ============================================================
# Compare v10 event-level ratio vs v11 resource-level ratio
# ============================================================

COMPARISON = (
    V10_METRICS[
        [
            "variant",
            "source_mapping_coverage",
        ]
    ]
    .rename(
        columns={
            "source_mapping_coverage":
                "source_mapping_coverage_v10_event_ratio"
        }
    )
    .merge(
        SOURCE_MAPPING_V11,
        on="variant",
        how="inner",
        validate="one_to_one",
    )
)


COMPARISON["absolute_change"] = (
    COMPARISON[
        "source_mapping_coverage_corrected"
    ]
    -
    COMPARISON[
        "source_mapping_coverage_v10_event_ratio"
    ]
)


COMPARISON["percentage_point_change"] = (
    COMPARISON["absolute_change"]
    * 100
)


display(COMPARISON)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if len(COMPARISON) != 6:
    raise RuntimeError(
        f"Expected 6 variants after merge, found {len(COMPARISON)}."
    )


if COMPARISON["variant"].nunique() != 6:
    raise RuntimeError(
        "Variant merge is not one-to-one."
    )


if not (
    COMPARISON[
        "source_mapping_coverage_corrected"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
):
    raise RuntimeError(
        "Corrected source mapping coverage "
        "is outside [0, 1]."
    )


if not (
    COMPARISON[
        "source_resources_mapped"
    ]
    <=
    COMPARISON[
        "source_resources_total"
    ]
).all():
    raise RuntimeError(
        "Mapped source-resource count "
        "exceeds denominator."
    )


if np.allclose(
    COMPARISON[
        "source_mapping_coverage_corrected"
    ],
    COMPARISON[
        "source_mapping_coverage_v10_event_ratio"
    ],
):
    raise RuntimeError(
        "Resource-level correction did not change "
        "the previous event-level ratio."
    )


print(
    "PASS: v10 event-level denominator and "
    "v11 resource-level denominator compared successfully."
)

,variant,source_mapping_coverage_v10_event_ratio,source_resources_total,source_resources_mapped,source_resources_unmapped,multi_event_source_resources,source_mapping_coverage_corrected,absolute_change,percentage_point_change
0,V0,0.842405,154333,126071,28262,25000,0.816876,-0.025528,-2.552842
1,V1,0.842405,154333,126071,28262,25000,0.816876,-0.025528,-2.552842
2,V2,0.842405,154333,126071,28262,25000,0.816876,-0.025528,-2.552842
3,V3,0.842405,154333,126071,28262,25000,0.816876,-0.025528,-2.552842
4,V4,0.855892,151507,126071,25436,25000,0.832113,-0.023779,-2.377903
5,V5,0.842405,154333,126071,28262,25000,0.816876,-0.025528,-2.552842


PASS: v10 event-level denominator and v11 resource-level denominator compared successfully.


# Phase C

## Corrected four primary metrics

In [8]:
# ============================================================
# Phase C — Corrected four primary metrics
# ============================================================

PRIMARY_METRICS_V11 = (
    V10_METRICS[
        [
            "variant",
            "lineage_coverage",
            "transformation_loss_rate",
            "ambiguity_warning_rate",
        ]
    ]
    .merge(
        SOURCE_MAPPING_V11[
            [
                "variant",
                "source_resources_total",
                "source_resources_mapped",
                "source_mapping_coverage_corrected",
            ]
        ],
        on="variant",
        how="inner",
        validate="one_to_one",
    )
    .rename(
        columns={
            "source_mapping_coverage_corrected":
                "source_mapping_coverage"
        }
    )
)


# ------------------------------------------------------------
# Put columns in final manuscript order
# ------------------------------------------------------------

PRIMARY_METRICS_V11 = PRIMARY_METRICS_V11[
    [
        "variant",
        "source_resources_total",
        "source_resources_mapped",
        "lineage_coverage",
        "source_mapping_coverage",
        "transformation_loss_rate",
        "ambiguity_warning_rate",
    ]
]


display(PRIMARY_METRICS_V11)


# ============================================================
# Validate that only source mapping coverage changed
# ============================================================

CHECK = (
    PRIMARY_METRICS_V11[
        [
            "variant",
            "lineage_coverage",
            "transformation_loss_rate",
            "ambiguity_warning_rate",
        ]
    ]
    .merge(
        V10_METRICS[
            [
                "variant",
                "lineage_coverage",
                "transformation_loss_rate",
                "ambiguity_warning_rate",
            ]
        ],
        on="variant",
        how="inner",
        suffixes=("_v11", "_v10"),
        validate="one_to_one",
    )
)


if len(CHECK) != 6:
    raise RuntimeError(
        f"Expected 6 variants in metric validation, found {len(CHECK)}."
    )


for metric in [
    "lineage_coverage",
    "transformation_loss_rate",
    "ambiguity_warning_rate",
]:

    if not np.allclose(
        CHECK[f"{metric}_v11"],
        CHECK[f"{metric}_v10"],
        equal_nan=True,
    ):

        raise RuntimeError(
            f"{metric} changed unexpectedly."
        )


# ============================================================
# Validate corrected source mapping metric
# ============================================================

if not (
    PRIMARY_METRICS_V11[
        "source_mapping_coverage"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
):
    raise RuntimeError(
        "Source mapping coverage is outside [0, 1]."
    )


if not (
    PRIMARY_METRICS_V11[
        "source_resources_mapped"
    ]
    <=
    PRIMARY_METRICS_V11[
        "source_resources_total"
    ]
).all():
    raise RuntimeError(
        "Mapped source-resource count exceeds total source resources."
    )


print(
    "PASS: corrected four primary metrics generated."
)

print(
    "PASS: lineage coverage, transformation-loss rate, "
    "and ambiguity/warning rate are unchanged from v10."
)

print(
    "PASS: only source mapping coverage uses the corrected "
    "resource-level denominator."
)

,variant,source_resources_total,source_resources_mapped,lineage_coverage,source_mapping_coverage,transformation_loss_rate,ambiguity_warning_rate
0,V0,154333,126071,1.00000,0.816876,0.157595,0.211612
1,V1,154333,126071,1.00000,0.816876,0.157595,0.212510
2,V2,154333,126071,0.98416,0.816876,0.170939,0.224956
3,V3,154333,126071,1.00000,0.816876,0.157595,0.225608
4,V4,151507,126071,1.00000,0.832113,0.144108,0.199873
5,V5,154333,126071,1.00000,0.816876,0.157595,0.226506


PASS: corrected four primary metrics generated.
PASS: lineage coverage, transformation-loss rate, and ambiguity/warning rate are unchanged from v10.
PASS: only source mapping coverage uses the corrected resource-level denominator.


## Manuscript table and export

In [9]:
MANUSCRIPT_PRIMARY_METRICS_V11 = PRIMARY_METRICS_V11.copy()

for column in [
    "lineage_coverage",
    "source_mapping_coverage",
    "transformation_loss_rate",
    "ambiguity_warning_rate",
]:
    MANUSCRIPT_PRIMARY_METRICS_V11[column] = (
        100.0
        * MANUSCRIPT_PRIMARY_METRICS_V11[column]
    ).round(2)

MANUSCRIPT_PRIMARY_METRICS_V11 = (
    MANUSCRIPT_PRIMARY_METRICS_V11.rename(
        columns={
            "source_resources_total":
                "Source resources, n",
            "source_resources_mapped":
                "Mapped source resources, n",
            "lineage_coverage":
                "Lineage coverage (%)",
            "source_mapping_coverage":
                "Source mapping coverage (%)",
            "transformation_loss_rate":
                "Transformation-loss rate (%)",
            "ambiguity_warning_rate":
                "Ambiguity/warning rate (%)",
        }
    )
)

display(MANUSCRIPT_PRIMARY_METRICS_V11)


SOURCE_MAPPING_V11.to_csv(
    V11_OUTPUT_DIR
    / "tfl_source_mapping_coverage_diagnostics_v11.csv",
    index=False,
)

COMPARISON.to_csv(
    V11_OUTPUT_DIR
    / "tfl_source_mapping_v10_vs_v11.csv",
    index=False,
)

PRIMARY_METRICS_V11.to_csv(
    V11_OUTPUT_DIR
    / "tfl_primary_metrics_v11.csv",
    index=False,
)

MANUSCRIPT_PRIMARY_METRICS_V11.to_csv(
    V11_OUTPUT_DIR
    / "tfl_primary_metrics_manuscript_v11.csv",
    index=False,
)

print("Saved outputs to:", V11_OUTPUT_DIR)

,variant,"Source resources, n","Mapped source resources, n",Lineage coverage (%),Source mapping coverage (%),Transformation-loss rate (%),Ambiguity/warning rate (%)
0,V0,154333,126071,100.00,81.69,15.76,21.16
1,V1,154333,126071,100.00,81.69,15.76,21.25
2,V2,154333,126071,98.42,81.69,17.09,22.50
3,V3,154333,126071,100.00,81.69,15.76,22.56
4,V4,151507,126071,100.00,83.21,14.41,19.99
5,V5,154333,126071,100.00,81.69,15.76,22.65


Saved outputs to: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/metric_fix_v11
